# Prior Sensitivity Analysis for Meridian MMM (Mocha Dataset)

This notebook implements the Week 3–5 base deliverable for Project 1:
a prior sensitivity analysis using Google Meridian.

We study how changing the assumed ROI prior affects posterior ROI estimates
across media channels, holding all other modeling assumptions fixed.


<a name="install"></a>
## Step 0: Install and Enviroment Configuration

Install the latest version of Meridian, and verify that GPU is available.

In [ ]:
# Install meridian: from PyPI @ latest release
!pip install --upgrade google-meridian[colab,and-cuda,schema]

# Install meridian: from PyPI @ specific version
# !pip install google-meridian[colab,and-cuda,schema]==1.3.1

# Install meridian: from GitHub @HEAD
# !pip install --upgrade "google-meridian[colab,and-cuda,schema] @ git+https://github.com/google/meridian.git@main"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.3/363.3 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 895.7/895.7 kB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 577.2/577.2 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.5/192.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.3/130.3 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.6/217.6 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.0/199.0 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.analysis.review import reviewer
from meridian.data import data_frame_input_data_builder
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
from schema.serde import meridian_serde
import numpy as np
import pandas as pd
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp
import matplotlib.pyplot as plt
import warnings
import time
import gc
import os

ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print(
    'Num GPUs Available: ',
    len(tf.config.experimental.list_physical_devices('GPU')),
)
print(
    'Num CPUs Available: ',
    len(tf.config.experimental.list_physical_devices('CPU')),
)

In [ ]:
dir(summarizer)


3\. Mount a storage. Use `meridian_root` to refer the mounted root. The mounted root will be used to <a href="#save-model">save trained model</a>, <a href="#generate-summary">stage two-pager output</a> and <a href="#scenario-planning">generate scenario planning dashboard</a>.

For Colab Free user, we will use the `MyDrive` folder in Google Drive as the external storage.

In [ ]:
# @markdown If you are using Colab Free, Colab Pro, run this cell to mount your Google Drive.
from google.colab import drive
drive_mount = '/content/drive'
drive.mount(drive_mount, force_remount=True)
subfolder = '' # @param {"type":"string","placeholder": "Optional, specifying a subfolder is recommended for organizing distinct execution runs."}
# Change this "MyDrive" to other share folders name if you would like to use a different drive.
meridian_root = f'{drive_mount}/MyDrive/{subfolder}'
is_enterprise_user=False

<a name="load-data"></a>
## Step 1: Load the data

1\. Read the data into a Pandas DataFrame.

In [ ]:
# Load Mocha simulated marketing data used for prior sensitivity analysis
df = pd.read_csv("monthly_mocha.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.rename(columns={"date": "time"})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Construct Meridian InputData

We construct a national-level InputData object containing KPI and media spend
for all advertising channels.


2\. Create a DataFrameInputDataBuilder instance.



In [ ]:
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type="non_revenue",
    default_kpi_column="subscriptions"
)

3\. Offer the components to the builder. Note that the components may be offered all at once or piecewise.

In [ ]:
print(sorted(df.columns))


In [ ]:
builder = builder.with_kpi(df)
channels = ["meta","google","snapchat"]
#"tiktok","moloco", "liveintent", "beehiiv", "amazon"
builder = builder.with_media(
    df,
    media_cols=[f"{c}_impressions" for c in channels],
    media_spend_cols=[f"{c}_spend" for c in channels],
    media_channels=channels,
)
channels = list(channels)  # freeze ordering
N_CHANNELS = len(channels)

4. Finally, build the InputData.

In [ ]:
data = builder.build()

<a name="configure-model"></a>
## Step 2: Configure the model

### Model Specification

We define a function that builds a Meridian model specification given ROI prior
parameters. This allows us to systematically vary ROI assumptions while keeping
all other model components fixed.

In [ ]:
# Step 2A: ModelSpec builder (THIS replaces the old Step 2)
BASE_ROI_MU = 0.4
BASE_ROI_SIGMA = 0.5

def build_model_spec(target_channel, roi_mu, roi_sigma):
    roi_mu_vec = np.full(
        len(channels),
        BASE_ROI_MU,
        dtype=np.float32
    )
    roi_sigma_vec = np.full(
        len(channels),
        BASE_ROI_SIGMA,
        dtype=np.float32
    )

    idx = channels.index(target_channel)
    roi_mu_vec[idx] = np.float32(roi_mu)
    roi_sigma_vec[idx] = np.float32(roi_sigma)

    roi_prior = tfp.distributions.LogNormal(
        loc=roi_mu_vec,
        scale=roi_sigma_vec,
        name="roi_m"
    )

    prior = prior_distribution.PriorDistribution(
        roi_m=roi_prior
    )

    return spec.ModelSpec(
        prior=prior,
        enable_aks=True
    )


In [ ]:
def extract_roi_mean(mmm, channels):
    """
    Extract posterior mean ROI per channel from a fitted Meridian model.
    Assumes Meridian stores posterior as ArviZ InferenceData (inference_data).
    """

    # Make sure posterior exists
    if not hasattr(mmm, "inference_data"):
        raise AttributeError(
            "No posterior found on mmm. "
            "Did you run mmm.sample_posterior()?"
        )

    idata = mmm.inference_data

    # Check ROI variable
    if "roi_m" not in idata.posterior:
        raise KeyError(
            f"'roi_m' not found in posterior variables: "
            f"{list(idata.posterior.data_vars)}"
        )

    # Extract ROI samples
    roi = idata.posterior["roi_m"]

    # Average over chains and draws
    mean_roi = roi.mean(dim=["chain", "draw"]).values

    # Sanity check
    if len(mean_roi) != len(channels):
        raise ValueError(
            "ROI length does not match number of channels."
        )

    return pd.DataFrame(
        {"roi_mean": mean_roi},
        index=channels
    )


In [ ]:
warnings.filterwarnings("ignore")
roi_mu_values = [0.2, 0.4, 0.6, 0.8]
output_file = "prior_sensitivity_results.csv"

results = []

# If file exists, load previous results (resume-safe)
if os.path.exists(output_file):
    print("Existing results found. Loading...")
    results = pd.read_csv(output_file).to_dict("records")

# Resume-safe skipping setup
already_done = set(
    (r["target_channel"], r["roi_prior_mu"])
    for r in results
)

total_runs = len(channels) * len(roi_mu_values)
run_id = 1

for target_channel in channels:
    for mu in roi_mu_values:

        print(f"\n===== Run {run_id}/{total_runs} =====")
        print(f"Channel: {target_channel}, Prior mu: {mu}")

        start_time = time.time()
        mmm = None  # safeguard

        try:

            model_spec = build_model_spec(
                target_channel=target_channel,
                roi_mu=mu,
                roi_sigma=BASE_ROI_SIGMA
            )

            mmm = model.Meridian(
                input_data=data,
                model_spec=model_spec
            )

            print("Sampling posterior...")
            mmm.sample_posterior(
                n_chains=1,
                n_adapt=100,
                n_burnin=50,
                n_keep=20,
                seed=0
            )
            print("Sampling finished.")

            roi_df = extract_roi_mean(mmm, channels)

            for ch in channels:
                results.append({
                    "target_channel": target_channel,
                    "roi_prior_mu": float(mu),
                    "channel": ch,
                    "estimated_roi": float(roi_df.loc[ch, "roi_mean"])
                })

            # SAVE AFTER EACH RUN
            pd.DataFrame(results).to_csv(output_file, index=False)
            print("Results saved.")

        except Exception as e:
            print(f"Run failed: {e}")

        end_time = time.time()
        print(f"Run completed in {(end_time - start_time):.2f} seconds")

        # Safe cleanup
        if mmm is not None:
            del mmm
        gc.collect()

        run_id += 1

print("\nAll runs finished.")
print(f"Total results collected: {len(results)}")

In [ ]:
results_df = pd.read_csv(output_file)
unique_channels = results_df["channel"].unique()

n_channels = len(unique_channels)
fig, axes = plt.subplots(
    nrows=int(np.ceil(n_channels / 2)),
    ncols=2,
    figsize=(12, 4 * int(np.ceil(n_channels / 2)))
)

axes = axes.flatten()

for i, ch in enumerate(unique_channels):

    ch_df = results_df[
        (results_df["target_channel"] == ch) &
        (results_df["channel"] == ch)
    ]

    # Sort before plotting
    ch_df = ch_df.sort_values("roi_prior_mu")

    axes[i].plot(
        ch_df["roi_prior_mu"],
        ch_df["estimated_roi"],
        marker="o"
    )

    axes[i].set_title(f"{ch} ROI Sensitivity")
    axes[i].set_xlabel("ROI Prior Mean")
    axes[i].set_ylabel("Estimated ROI")

plt.tight_layout()
plt.show()

In [ ]:
results_df = pd.read_csv("prior_sensitivity_results.csv")

# Detect baseline automatically (middle prior if symmetric)
unique_mus = sorted(results_df["roi_prior_mu"].unique())
baseline_mu = 0.4

print("\nSENSITIVITY REPORT")
print("Most Sensitive Parameters:\n")

report_rows = []

for ch in results_df["channel"].unique():

    ch_df = results_df[
        (results_df["target_channel"] == ch) &
        (results_df["channel"] == ch)
    ].sort_values("roi_prior_mu")

    if ch_df.empty:
        continue

    # Prior range
    min_mu = ch_df["roi_prior_mu"].min()
    max_mu = ch_df["roi_prior_mu"].max()

    prior_down_pct = (min_mu - baseline_mu) / baseline_mu * 100
    prior_up_pct = (max_mu - baseline_mu) / baseline_mu * 100

    # Posterior range
    roi_min = ch_df["estimated_roi"].min()
    roi_max = ch_df["estimated_roi"].max()

    baseline_roi = ch_df[
        ch_df["roi_prior_mu"] == baseline_mu
    ]["estimated_roi"].values[0]

    posterior_change_pct = (
        (roi_max - roi_min) / baseline_roi
    ) * 100

    report_rows.append({
        "channel": ch,
        "prior_down_pct": prior_down_pct,
        "prior_up_pct": prior_up_pct,
        "posterior_change_pct": posterior_change_pct
    })

# Rank by posterior sensitivity
report_df = pd.DataFrame(report_rows).sort_values(
    "posterior_change_pct", ascending=False
)

# Print report
for _, row in report_df.iterrows():

    ch = row["channel"]
    prior_down = abs(row["prior_down_pct"])
    prior_up = abs(row["prior_up_pct"])
    posterior_pct = row["posterior_change_pct"]

    robustness_tag = ""
    if posterior_pct < 5:
        robustness_tag = " (robust)"
    elif posterior_pct > 15:
        robustness_tag = " (high sensitivity)"

    print(
        f"{ch.upper()} ROI prior: "
        f"-{prior_down:.0f}% to +{prior_up:.0f}% change → "
        f"{posterior_pct:.1f}% change in estimated ROI"
        f"{robustness_tag}"
    )


<a name="quality-checks"></a>
## Step 3: Run post-modeling quality checks

These post-modeling quality checks are designed to diagnose common issues related to model convergence, specification, and plausibility. Run the following command to generate the results for all necessary diagnostics:

In [ ]:
reviewer.ModelReviewer(mmm).run()

<a name="model-diagnostics"></a>
## Step 4: Run model diagnostics

To further assess convergence and model fit, you can use the methods from `visualizer` module.

1\. Assess convergence. Run the following code to generate r-hat statistics. R-hat close to 1.0 indicate convergence. R-hat < 1.2 indicates approximate convergence and is a reasonable threshold for many problems.

In [ ]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()

2\. Assess the model's fit by comparing the expected sales against the actual sales.

In [ ]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

For more information and additional model diagnostics checks, see [Modeling diagnostics](https://developers.google.com/meridian/docs/user-guide/model-diagnostics).

<a name="generate-summary"></a>
## Step 5: Generate model results & two-page output

To export the two-page HTML summary output, initialize the `Summarizer` class with the model object. Then pass in the filename, filepath, start date, and end date to `output_model_results_summary` to run the summary for that time duration and save it to the specified file.

In [ ]:
mmm_summarizer = summarizer.Summarizer(mmm)

In [ ]:
filepath = meridian_root
start_date = '2024-03-11'
end_date = '2025-08-04'
mmm_summarizer.output_model_results_summary(
    'summary_output.html', filepath, start_date, end_date
)

Here is a preview of the two-page output based on the simulated data:

In [ ]:
IPython.display.HTML(filename=f'{meridian_root}/summary_output.html')

For a customized two-page report, model results summary table, and individual visualizations, see [Model results report](https://developers.google.com/meridian/docs/user-guide/generate-model-results-report) and [plot media visualizations](https://developers.google.com/meridian/docs/user-guide/plot-media-visualizations).





<a name="generate-optimize"></a>
## Step 6: Run budget optimization & generate an optimization report

You can choose what scenario to run for the budget allocation. In default scenario, you find the optimal allocation across channels for a given budget to maximize the return on investment (ROI).

Alternatively, if you would like to have a sharable interactive dashboard, check out [Meridian Scenario Planner](https://developers.google.com/meridian/docs/scenario-planning/meridian-scenario-planner).

1\. Instantiate the `BudgetOptimizer` class and run the `optimize()` method without any customization, to run the default library's Fixed Budget Scenario to maximize ROI.

In [ ]:
%%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

2\. Export the 2-page HTML optimization report, which contains optimized spend allocations and ROI.

In [ ]:
filepath = meridian_root
optimization_results.output_optimization_summary(
    'optimization_output.html', filepath
)

In [ ]:
IPython.display.HTML(filename=f'{meridian_root}/optimization_output.html')

For information about customized optimization scenarios, such as flexible budget scenarios, see [Budget optimization scenarios](https://developers.google.com/meridian/docs/user-guide/budget-optimization-scenarios). For more information about optimization results summary and individual visualizations, see [optimization results output](https://developers.google.com/meridian/docs/user-guide/generate-optimization-results-output) and [optimization visualizations](https://developers.google.com/meridian/docs/user-guide/plot-optimization-visualizations).


Optimization can also be performed on a hypothetical data representing a future scenario. The new data takes the same structure as the input data and encodes an anticipated flighting pattern, cost per media unit, and revenue per kpi.

3\. Load the [simulated dataset in CSV format](https://github.com/google/meridian/blob/main/meridian/data/simulated_data/csv/hypothetical_geo_all_channels.csv) into Pandas DataFrame.

In [ ]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/google/meridian/refs/heads/main/meridian/data/simulated_data/csv/hypothetical_geo_all_channels.csv"
)



4\. New data is read from a csv file and converted into a set of multi-dimensional arrays. The arrays are used to construct a `DataTensors` instance, which is passed to `optimize()` as the `new_data` argument.

Constructing a `DataTensors` instance requires that all arrays have "time" and "geo" dimensions. Alternatively, the `BudgetOptimizer.create_optimization_tensors` method can be used to construct a `DataTensors` instance. This helper method can simplify the process, particularly when you do not need "time" and "geo" dimensions for all inputs. For example, it can be convenient if you want to assume a constant "revenue per kpi" or "cost per media unit" for all geos and time periods.

In [ ]:
n_geos = mmm.n_geos
n_media_channels = mmm.n_media_channels
n_non_media_channels = mmm.n_non_media_channels
n_organic_media_channels = mmm.n_organic_media_channels

# The number of time periods and time range do not need to match the input data.
df[constants.TIME] = pd.to_datetime(df[constants.TIME], errors='coerce')
unique_times = sorted(df[constants.TIME].unique())
n_times = len(unique_times)

geos = mmm.input_data.geo.values
media_channels = mmm.input_data.media_channel.values
media_cols = [f"{channel}_impression" for channel in media_channels]
media_spend_cols = [f"{channel}_spend" for channel in media_channels]
non_media_treatment_cols = ['Promo']
organic_media_cols = ['Organic_channel0_impression']
organic_media_channels = ['Organic_channel0']
revenue_per_kpi_col='revenue_per_conversion'
times_str = [time.strftime(constants.DATE_FORMAT) for time in unique_times]

media_np = np.zeros((n_geos, n_times, n_media_channels))
media_spend_np = np.zeros((n_geos, n_times, n_media_channels))
non_media_treatment_np = np.zeros((n_geos, n_times, n_non_media_channels))
organic_media_np = np.zeros((n_geos, n_times, n_organic_media_channels))
revenue_per_kpi_np = np.zeros((n_geos, n_times))

df_grouped = df.set_index([constants.GEO, constants.TIME])
for geo_idx, geo in enumerate(geos):
  for time_idx, time in enumerate(unique_times):
    row = df_grouped.loc[(geo, time)]
    media_np[geo_idx, time_idx, :] = row[media_cols].values
    media_spend_np[geo_idx, time_idx, :] = row[media_spend_cols].values
    non_media_treatment_np[geo_idx, time_idx, :] = row[non_media_treatment_cols].values
    organic_media_np[geo_idx, time_idx, :] = row[organic_media_cols].values
    revenue_per_kpi_np[geo_idx, time_idx] = row[revenue_per_kpi_col].item()

data_tensors = analyzer.DataTensors(
    media=tf.convert_to_tensor(media_np, dtype=tf.float32),
    media_spend=tf.convert_to_tensor(media_spend_np, dtype=tf.float32),
    non_media_treatments=tf.convert_to_tensor(non_media_treatment_np, dtype=tf.float32),
    organic_media=tf.convert_to_tensor(organic_media_np, dtype=tf.float32),
    revenue_per_kpi=tf.convert_to_tensor(revenue_per_kpi_np, dtype=tf.float32),
    time=tf.convert_to_tensor(times_str, dtype=tf.string),
)
# Default values for `budget` and `pct_of_spend` are derived from the `new_data`,
# but these values can be overridden without modifying the `new_data` itself.
hypothetical_optimization_results = budget_optimizer.optimize(
    new_data=data_tensors,
    budget=50_000_000,
    pct_of_spend=[.2, .1, .2, .2, .3]
)

5\. Export the 2-page HTML optimization report.

In [ ]:
filepath = meridian_root
hypothetical_optimization_results.output_optimization_summary(
    'hypothetical_optimization_output.html', filepath
)

In [ ]:
IPython.display.HTML(filename=f'{meridian_root}/hypothetical_optimization_output.html')

<a name="save-model"></a>
## Step 7: Save the model object

We recommend that you save the model object for future use. This helps you to  avoid repetitive model runs and saves time and computational resources. After the model object is saved, you can load it at a later stage to continue the analysis or visualizations without having to re-run the model.


Run the following codes to save the model object:

In [ ]:
file_path = f'{meridian_root}/saved_mmm.binpb'
meridian_serde.save_meridian(mmm, file_path)
print(f'model is saved at {file_path}')

Run the following codes to load the saved model:

In [ ]:
mmm = meridian_serde.load_meridian(file_path)

In [ ]:
# @markdown If a GCS bucket is mounted, run this cell to unmount it.
!fusermount -u /content/{bucket_name}